In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import re
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/training_data_binary.csv') # train dataset
test_df = pd.read_csv('/content/drive/MyDrive/testing_data_binary.csv') # test dataset
train_df.columns.values[-2] = 'prompt'
test_df.columns.values[-2] = 'prompt'
X_train = train_df['prompt']
y_train = train_df['jailbreak']
X_val = test_df['prompt']
y_val = test_df['jailbreak']

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [ ]:
from tqdm import tqdm
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
def init_metrics():
    return {
        'accuracy': [],
        'auc': [],
        'tpr': [],
        'fnr': []
    }
def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
    return acc, auc, tpr, fnr
results = {
    'LogisticRegression': init_metrics(),
    'ExtraTrees': init_metrics(),
    'LGBM': init_metrics()
}
for i in tqdm(range(30), desc="Running models"):
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_tfidf, y_train, test_size=0.2, stratify=y_train, random_state=i
    )
    model_LR = LogisticRegression(random_state=i, solver='saga', penalty='elasticnet', l1_ratio=0.2, max_iter=8000)
    model_LR.fit(X_train_split, y_train_split)
    y_pred = model_LR.predict(X_val_split)
    y_proba = model_LR.predict_proba(X_val_split)[:, 1]
    acc, auc, tpr, fnr = compute_metrics(y_val_split, y_pred, y_proba)
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], [acc, auc, tpr, fnr]):
        results['LogisticRegression'][key].append(val)
    model_ET = ExtraTreesClassifier(n_estimators=100, random_state=i)
    model_ET.fit(X_train_split, y_train_split)
    y_pred = model_ET.predict(X_val_split)
    y_proba = model_ET.predict_proba(X_val_split)[:, 1]
    acc, auc, tpr, fnr = compute_metrics(y_val_split, y_pred, y_proba)
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], [acc, auc, tpr, fnr]):
        results['ExtraTrees'][key].append(val)
    model_LGBM = LGBMClassifier(n_jobs=-1, random_state=i)
    model_LGBM.fit(X_train_split, y_train_split)
    y_pred = model_LGBM.predict(X_val_split)
    y_proba = model_LGBM.predict_proba(X_val_split)[:, 1]
    acc, auc, tpr, fnr = compute_metrics(y_val_split, y_pred, y_proba)
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], [acc, auc, tpr, fnr]):
        results['LGBM'][key].append(val)
def summarize_model(name, metrics):
    print(f"\n===== {name} =====")
    for metric in ['accuracy', 'auc', 'tpr', 'fnr']:
        mean = np.mean(metrics[metric])
        std = np.std(metrics[metric])
        print(f"{metric.upper():<5}: Mean = {mean:.4f}, Std = {std:.4f}")
for model_name in results:
    summarize_model(model_name, results[model_name])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm

In [ ]:
train_df = pd.read_csv('/content/testing_data_binary.csv')
test_df = pd.read_csv('/content/testing_data_binary.csv')

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
def extract_bert_features(texts, tokenizer, model, max_length=128):
    embeddings = []
    for text in tqdm(texts, desc="Extracting BERT Features"):
        tokens = tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        with torch.no_grad():
            output = model(
                input_ids=tokens['input_ids'],
                attention_mask=tokens['attention_mask']
            )
            embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
            embeddings.append(embedding)
    return np.array(embeddings)

In [ ]:
X_train = extract_bert_features(train_df['prompt'], tokenizer, bert_model)
X_test = extract_bert_features(test_df['prompt'], tokenizer, bert_model)
y_train = train_df['jailbreak']
y_test = test_df['jailbreak']

In [ ]:
from tqdm import tqdm
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
X_train = extract_bert_features(train_df['prompt'], tokenizer, bert_model)
X_test = extract_bert_features(test_df['prompt'], tokenizer, bert_model)
y_train = train_df['jailbreak']
y_test = test_df['jailbreak']
def init_metrics():
    return {'accuracy': [], 'auc': [], 'tpr': [], 'fnr': []}
def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
    return acc, auc, tpr, fnr
results = {
    'LogisticRegression': init_metrics(),
    'ExtraTrees': init_metrics(),
    'LGBM': init_metrics()
}
for i in tqdm(range(30), desc="Training models"):
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=i
    )
    model_LR = LogisticRegression(random_state=i, solver='saga', penalty='elasticnet', l1_ratio=0.2, max_iter=8000)
    model_LR.fit(X_train_split, y_train_split)
    y_pred = model_LR.predict(X_val_split)
    y_proba = model_LR.predict_proba(X_val_split)[:, 1]
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], compute_metrics(y_val_split, y_pred, y_proba)):
        results['LogisticRegression'][key].append(val)
    model_ET = ExtraTreesClassifier(n_estimators=100, random_state=i)
    model_ET.fit(X_train_split, y_train_split)
    y_pred = model_ET.predict(X_val_split)
    y_proba = model_ET.predict_proba(X_val_split)[:, 1]
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], compute_metrics(y_val_split, y_pred, y_proba)):
        results['ExtraTrees'][key].append(val)
    model_LGBM = LGBMClassifier(n_jobs=-1, random_state=i)
    model_LGBM.fit(X_train_split, y_train_split)
    y_pred = model_LGBM.predict(X_val_split)
    y_proba = model_LGBM.predict_proba(X_val_split)[:, 1]
    for key, val in zip(['accuracy', 'auc', 'tpr', 'fnr'], compute_metrics(y_val_split, y_pred, y_proba)):
        results['LGBM'][key].append(val)
def summarize_model(name, metrics):
    print(f"\n===== {name} =====")
    for metric in ['accuracy', 'auc', 'tpr', 'fnr']:
        mean = np.mean(metrics[metric])
        std = np.std(metrics[metric])
        print(f"{metric.upper():<5}: Mean = {mean:.4f}, Std = {std:.4f}")

for model_name in results:
    summarize_model(model_name, results[model_name])